# 4.6 — Using Text Files

**AP CSA · Unit 4: Data Collections**

A school club has saved its attendance numbers. How can a Java program use those numbers tomorrow without someone typing them again?

## Learning targets

By the end, you can connect a text file to a `Scanner`, choose the right reading method, trace a file-reading loop, and separate a simple delimited line into fields.

**Before you start:** variables, `while` loops, arrays, and String methods.

**Try it:** Predict the answer, record your work in your notes, and compare with a partner. Open each answer reveal after attempting the activity.


## 1. Warm-up: data that lasts

Imagine `int attendance = 24;` exists while your program runs. The program ends. Tomorrow it starts again.

1. Does that variable automatically remember yesterday's value?
2. How would saving `24` in a text file help?
3. Is a filename the same thing as the contents of a file?

<details markdown="1">
<summary>Check your reasoning</summary>

1. No. A local variable does not automatically preserve its value between program runs.
2. The file can remain on storage after the program exits. A later run can read it.
3. No. A filename/path tells Java where to look; the contents are the data stored there.

</details>


## 2. Read an attendance file

For optional execution, save this data as `attendance.txt` in your Java program's **working directory** (the directory from which it runs). That is not always the folder containing the source file.

```text
18 24
21 17
```

Save the complete program below as `AttendanceReader.java`.

```java
import java.io.File;
import java.io.IOException;
import java.util.Scanner;

public class AttendanceReader {
    public static void main(String[] args) throws IOException {
        File attendanceFile = new File("attendance.txt");
        Scanner input = new Scanner(attendanceFile);
        int total = 0;
        int days = 0;

        while (input.hasNext()) {
            int students = input.nextInt();
            total += students;
            days++;
        }
        input.close();

        System.out.println("Days: " + days);
        System.out.println("Total: " + total);
    }
}
```

**Walk through it together:**

- `File` represents the path. Creating the `File` object does not create a missing file or read its contents.
- `Scanner` opens that file for reading. Here we promise the file contains only valid integer tokens.
- `hasNext()` checks whether another token exists; it does **not** consume it or verify that it is an integer.
- `nextInt()` consumes one integer token. Spaces and line breaks separate tokens.
- `throws IOException` lets file-opening failures propagate to the caller. It does not repair a missing file; if uncaught in this program, the failure stops execution.
- `close()` releases the scanner's resources when reading is complete.

**Think about it:** Point to the exact statement that moves the scanner forward. What happens if we remove it but leave the loop condition?

<details markdown="1">
<summary>Check your reasoning</summary>

`input.nextInt()` consumes input. Without any consuming read inside the loop, `hasNext()` keeps reporting the same available token, so a nonempty file can cause an infinite loop.

</details>


## 3. Partner activity: be the scanner

One partner points to the next unread token. The other updates the variables. Swap roles after two iterations.

**Predict:** Does the line break after `24` change the total?

| Iteration | Token consumed | total after addition | days after increment |
|---|---|---|---|
| 1 | 18 | … | … |
| 2 | 24 | … | … |
| 3 | 21 | … | … |
| 4 | 17 | … | … |

<details markdown="1">
<summary>Check your reasoning</summary>

| Iteration | Token | total | days |
|---|---|---|---|
| 1 | 18 | 18 | 1 |
| 2 | 24 | 42 | 2 |
| 3 | 21 | 63 | 3 |
| 4 | 17 | 80 | 4 |

The output is `Days: 4` followed by `Total: 80`. The newline separates tokens just like a space here. After the fourth read, `hasNext()` is false.

**Change the input:** An empty file produces `Days: 0` and `Total: 0`. A file containing `18 absent 21` fails at `nextInt()` when the next token is `absent`; `hasNext()` alone does not guarantee a numeric token.

</details>


### Your response

- My predicted total:
- The statement that consumes data:
- What changes if I add `30` on a new line:
- My explanation of the empty-file result:


## 4. Choose the reading method

Each row below is an **independent** reading situation; assume the scanner is at the start of the shown input.

| Input | What you want | Method | Result type |
|---|---|---|---|
| `24` | One integer | `nextInt()` | `int` |
| `3.5` | One decimal value | `nextDouble()` | `double` |
| `true` | One Boolean value | `nextBoolean()` | `boolean` |
| `Robotics Club` | The first word | `next()` | `String` |
| `Robotics Club` | The whole line | `nextLine()` | `String` |
| Any remaining token | Whether something is available | `hasNext()` | `boolean` |

**Quick vote:** A club name contains spaces. Should we read the name with `next()` or `nextLine()`? Explain the information we lose with the other choice.

For this lesson, use separate scanners/examples for token reading and line reading. Mixing `nextLine()` with token-reading methods on one input source is outside the specified AP exam scope for this topic.

### Turn one line into fields

Suppose `clubs.txt` has exactly these two nonblank lines, with one comma per line and no embedded commas:

```text
Robotics,Room 12
Art,Room 8
```

Inside a method declared with `throws IOException`, and with the earlier imports:

```java
Scanner clubs = new Scanner(new File("clubs.txt"));
while (clubs.hasNext()) {
    String line = clubs.nextLine();
    String[] fields = line.split(",");
    System.out.println(fields[0] + " meets in " + fields[1]);
}
clubs.close();
```

`split(",")` creates a String array. Array indexing still starts at zero. This example assumes every line follows the two-field format; arbitrary CSV files can be more complicated.

**Predict:** What is `fields.length` for the first record? What is printed?

<details markdown="1">
<summary>Check your reasoning</summary>

`fields.length` is `2`. The two output lines are:

```text
Robotics meets in Room 12
Art meets in Room 8
```

The name example needs `nextLine()`: `next()` would return only `Robotics`. The club loop uses only line reads; `hasNext()` checks availability without consuming input. Its stated nonblank, consistently formatted input makes this pattern suitable here.

</details>


## 5. Debug before you code

A classmate writes this loop for a file of integers:

```java
while (input.hasNext()) {
    input.nextInt();
    total += input.nextInt();
}
```

Talk through `18 24 21 17`. Which values contribute to `total`? What if a fifth value is appended?

<details markdown="1">
<summary>Check your reasoning</summary>

The first read in each iteration discards a number. Only `24` and `17` are added, for `41`. With an odd number of tokens, the final iteration consumes the last token with the first call and then tries to read past the end with the second call. Fix this by reading once into a variable and using that variable.

</details>


## 6. Popcorn hack: count busy days

Write a method `countBusyDays(String filename) throws IOException` that returns how many integers in a file are at least `20`.

**Contract:** The file exists, contains only valid integer tokens, and may be empty. Read every token once. Close the scanner. Do not assume a fixed number of days.

Use the imports from the complete program and place this method inside a class.

```java
public static int countBusyDays(String filename) throws IOException {
    // Open a scanner.
    // Count values >= 20.
    // Close the scanner and return the count.
}
```

| File contents | Expected return |
|---|---|
| `18 24 21 17` | `2` |
| `20` | `1` |
| `19 0` | `0` |
| Empty file | `0` |

<details markdown="1">
<summary>Reveal a solution after writing yours</summary>

```java
public static int countBusyDays(String filename) throws IOException {
    Scanner input = new Scanner(new File(filename));
    int busyDays = 0;
    while (input.hasNext()) {
        int attendance = input.nextInt();
        if (attendance >= 20) {
            busyDays++;
        }
    }
    input.close();
    return busyDays;
}
```

Self-check: Did you open the provided filename, consume one value per iteration, include exactly `20`, close the scanner, and return zero for an empty file?

</details>


### Your solution

```java
// Write countBusyDays here.
```

My trace for the boundary value `20`:

One test I would add and its expected result:


## 7. Exit ticket

Answer individually before opening the key.

1. What is the difference between `hasNext()` and `nextInt()`?
2. Why might `new Scanner(new File("attendance.txt"))` fail even when the code compiles?
3. Given `String record = "Meryl,Blue";`, what does `record.split(",")[1]` produce?

<details markdown="1">
<summary>Exit-ticket key</summary>

1. `hasNext()` checks availability without advancing. `nextInt()` consumes a token as an integer and can fail if the token is invalid or unavailable.
2. The path may not locate a readable file in the working directory.
3. The String `"Blue"`.

</details>
